### MEG Workshop: Intro to Multivariate Methods
Code written by Lina Teichmann, April 2025

___
The main purpose of this notebook is to demonstrate the method and facilite learning. Feel free to make it all nicer by wrapping things in functions and streamlining the code to make it more efficient. 

<span style="color:SteelBlue">**Setup: Import packages & define paths**

In [ ]:
import os
import mne,itertools
import pandas as pd
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as lda
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score,GroupKFold
from scipy.ndimage import convolve1d


# helper function to run classification with LDA
def pipeline():
    return Pipeline([('scaler', StandardScaler()), ('clf', lda(solver='eigen', shrinkage=0.01))])

def train_lda(X_train, y_train):
    return pipeline().fit(X_train, y_train)

def test_lda(pipe,X_test,y_test,res_type='acc'):
    if res_type=='acc':
        res = pipe.score(X_test,y_test)
    elif res_type=='pred':
        pred = pipe.predict(X_test)
        res = np.stack([pred,y_test],axis=0)
    return res


<span style="color:Orchid">**Intro to Scikit-learn using the iris dataset**

You can skip this if you are already familiar with sklearn and just want to jump in the MEG-decoding!

<img src="./iris-machinelearning.png" alt="isolated" width="400"/>


The iris dataset is a standard machine learning dataset that is included in sklearn. I will use it below to introduce the ideas used for the MEG data with a simpler example. (If you are interested in learning more about modelling with sklearn, I highly recomment the scikit-learn MOOC course. It's free and really great!)


The goal here is to predict the flower type (as shown in the image above) from sepal & petal length & width. The thing you want to predict is called the *target* [setosa, versicolor, virginica] and you are using the *features* [sepal length, sepal width, petal length, petal width] to make your prediction. The model is typically trained on some data and tested on held-out data. 

In [ ]:
from sklearn import datasets

# load data and see what it contains
iris = datasets.load_iris()
print('***************')
print('Here are all dictonary keys showing you what is inside the dataset:')
print('***************')
for key in iris:
    print(key)

X = iris.data[:,0:2] # X is the data (features) -- only taking two features for easier visualization
y = iris.target # y are the classes (targets)
cv = np.tile(np.arange(50),3) # cv is a cross-validation variable (in this case a leave-one-out)
target_names = iris.target_names
feature_names = iris.feature_names
n_features = X.shape[1]

print('\n\n***************')
print('Here is a visualization how the different classes differ in terms of the first two features')
print('***************')
plt.figure()
for color, i, target_name in zip(["navy", "turquoise", "darkorange"], [0, 1, 2], target_names):
    plt.scatter(
        X[y == i, 0], X[y == i, 1], color=color, alpha=0.8, lw=2, label=target_name
    )
plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.legend(loc="best", shadow=False, scatterpoints=1)
plt.show()

# run multi-class classification: this is looping over the cross-validation splits and fits (trains) the model and then predicts (tests) the model
acc=[]
for cv_i in np.unique(cv):
    x_train = X[cv!=cv_i,:]
    x_test = X[cv==cv_i,:]
    y_train = y[cv!=cv_i]
    y_test = y[cv==cv_i]

    pipe = train_lda(x_train, y_train)

    acc.append(test_lda(pipe,x_test,y_test,res_type='acc'))

print('\n\n***************')
print(f'Cross-validated classification accuracy: {np.mean(acc)*100} % (chance-level is {np.round(100/len(np.unique(y)),2)}%)')
print('***************')

# running pairwise classifications
print('\n\n***************')
print('Pairwise classifications to look at the representational dissimilarity matrix')
print('***************')

# iterate over all unique pairs of classes and run the decoding
rdm = np.zeros((len(np.unique(y)),len(np.unique(y))))
for (i, class1), (j, class2) in itertools.combinations(enumerate(np.unique(y)), 2):
    mask = (y == class1) | (y == class2)
    X_pair = X[mask,:]
    y_pair = y[mask]
    cv_pair = cv[mask]

    # run over cross-validation
    acc=[]
    for cv_i in np.unique(cv):
        x_train = X_pair[cv_pair!=cv_i,:]
        x_test = X_pair[cv_pair==cv_i,:]
        y_train = y_pair[cv_pair!=cv_i]
        y_test = y_pair[cv_pair==cv_i]

        pipe = train_lda(x_train, y_train)

        acc.append(test_lda(pipe,x_test,y_test,res_type='acc'))
        
    
    rdm[class1,class2] = np.mean(acc)
np.fill_diagonal(rdm, np.nan)


# make a plot to show RDM
plt.imshow(rdm.T+rdm,cmap='magma',vmin=0.5,vmax=1)
plt.xticks(np.unique(iris.target))
plt.yticks(np.unique(iris.target))
plt.gca().set_xticklabels(np.unique(iris.target_names))
plt.gca().set_yticklabels(np.unique(iris.target_names))
plt.colorbar(label='decoding accuracy')
plt.show()


<span style="color:SteelBlue">**MEG DATA: Load epoched data**


In [ ]:
userid = os.environ['USER']
bids_dir = f'/data/{userid}/meg_workshop_2025adv/Decoding/bids_dir'
subjid = 'S03'

epochs = mne.read_epochs(f"{bids_dir}/derivatives/meg/preprocessed/sub-{subjid}_preprocessed-epo.fif")

# remove target trials (repeated images that required a button press)
epochs = epochs[epochs.metadata['im_class']!='target']

# addng cross-validation index to do a half-way cross validation instead of run-wise (faster)
epochs.metadata['cv_idx']=0
epochs.metadata.loc[epochs.metadata['run']<5,'cv_idx']=1

# plot averaged data
epochs.average().plot()

# display metadata
print(epochs.metadata)

# variables
n_timepoints = epochs._data.shape[2]
n_channels = epochs._data.shape[1]
n_trials = epochs._data.shape[0]

<span style="color:MediumVioletRed">**Challenge 1: Time-resolved category decoding with split-half cross-validation**.

Is there a reliable representations of category (i.e., objects and people) in the MEG data when participants viewed (a) control images and (b) anagram images? 
___
>__Train and test a model to distinguish objects and people__\
>✨ This needs to be done separately for control and anagram trials\
>✨ You need to train and test the model at EACH timepoint\
>✨ We need to cross-validate (e.g., split-half) to ensure we are training and testing on independent data\



In [ ]:
### STARTING POINT CHALLENGE 1:
# select only control images
selected_epochs = epochs[epochs.metadata['im_class']=='control']

# data (features) are the MEG data over time
X = selected_epochs._data
# class label (target) is the trial information whether a person or an object was shown
label_encoder = LabelEncoder() # this essentially makes the labels into 1s and 0s
y = label_encoder.fit_transform(selected_epochs.metadata['im_category'])
# cross-validation to train and test on independent data just using split-half for now
cv = selected_epochs.metadata['cv_idx']


# loop over cv-splits


# split training and testing set for cv-split


# run classification at every timepoint & save accuracy


# average classification accuracies across cv-splits


# plot accuracy over time (epochs.times has the time vector)



# repeat same thing for anagrams and see how the two compare

<span style="color:MediumVioletRed">**Challenge 2: Time-resolved category decoding across stimulus type**.

Does category information from control images generalize to anagrams despite matched low-level features?

___
>__Train a model on control image trials to distinguish objects and people and test on selected anagrams__\
>✨ contrast trials that used the same anagram but presented upright versus inverted (you can select different anagrams below and look how the plot changes)\
>✨ think about what above- and below-chance decoding means in this context\
>✨ come up with an intuitive way to plot the results


In [ ]:
# Train the model on control images
label_encoder = LabelEncoder()

# training set: select control stimuli and use label encoder to set y_train to be 0s and 1s instead of im_category. 
training_set = epochs[epochs.metadata['im_class']=='control']
y_train = label_encoder.fit_transform(training_set.metadata['im_category'])
x_train = training_set._data

# train the model
trained_model = [train_lda(x_train[:,:,t], y_train) for t in tqdm(range(x_train.shape[2]))]


In [ ]:
# Test model on selected anagram

# optional: look at which anagram you want to classify (default 6a is a good anagram for this person)
im_num = 6 # 1-12
exemplar = 'a' # a or b
ims_path = []
ims_path.append(f'{bids_dir}/sourcedata/stimuli/anagram_object_{exemplar}_{str(im_num).zfill(2)}.png')
ims_path.append(f'{bids_dir}/sourcedata/stimuli/anagram_person_{exemplar}_{str(im_num).zfill(2)}.png')

fig,axs = plt.subplots(1,2)
for i in range(2):
    im = plt.imread(ims_path[i])
    axs[i].imshow(im)
    axs[i].axis('off')
fig.suptitle(f'Anagram {str(im_num).zfill(2)}{exemplar}')

# Testing set: select anagram and get evoked data
# The anagram is defined by "im_num" and "exemplar"
# The label (y_test) is still im_category.
testing_set = epochs[epochs.metadata['im_class']=='anagram']
x_test = testing_set[(testing_set.metadata['im_number']==im_num)&(testing_set.metadata['im_exemplar']==exemplar)]._data
y_test = label_encoder.fit_transform(testing_set.metadata[(testing_set.metadata['im_number']==im_num)&(testing_set.metadata['im_exemplar']==exemplar)]['im_category'])

# test the model: set res_type='pred' to get the predicitions instead of the accuracy
res = [test_lda(trained_model[t],x_test[:,:,t],y_test,res_type='pred') for t in range(x_train.shape[2])]


# tease apart the results for anagram "object" and anagram "person" [hint: dimension 1 in res captures the true class]
# you can use label_encoder.classes_ to get the class order
true_class_idx_0 = np.array(res)[0,1,:]==0
correct_predictions_class0 = np.array(res)[:,0,true_class_idx_0]==0

true_class_idx_1 = np.array(res)[0,1,:]==1
correct_predictions_class1 = np.array(res)[:,0,true_class_idx_1]==1


In [ ]:
# how could we plot these results? 


<span style="color:MediumVioletRed">**Bonus content: Single timepoint representational dissimilarity matrix (RDM)**.

Generate a representational dissimilarity matrix (RDM) to summarize how well different image categories (i.e., person versus object) can be decoded within and across image types (i.e., anagrams versus controls).

Note: This problem could be run on the image level (i.e., comparing all 96 images with each other) which is typically what is done -- it just takes a lot longer.

___
>✨Think about what the RDM pattern means.\
>✨Generate a RDM at a different timepoint and see whether the results look as you would expect. 


In [ ]:
# select one timepoint of interest (250-300ms is quite good)
t_min = np.where(epochs.times==0.25)[0][0]
t_max = np.where(epochs.times==0.3)[0][0]
X = np.mean(epochs._data[:,:,t_min:t_max],axis=2)

# get the classes/labels for the decoding problem: 
# they are a combination of image class (anagram v control) and image category (object vs person)
rdm_class = epochs.metadata['im_class']+'_'+epochs.metadata['im_category']
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(rdm_class.to_numpy())
classes = np.unique(y)
n_classes = len(classes)

# we are using a sklearn function to run the cross-validation which needs the cross-validation groups to be specified like this [more efficient]
groups = epochs.metadata['run'].to_numpy()
group_kfold = GroupKFold(n_splits=len(np.unique(groups)))

# initalize empty RDM
rdm = np.zeros((n_classes, n_classes))

# iterate over all unique pairs of classes and run the decoding (cross-validated over runs)
# for (i, class1), (j, class2) in tqdm(itertools.combinations(enumerate(classes), 2)):
#       ....
#       ....


# plot results
